# 01 — SQL Analytics: Business Questions on Churn

Every question in this notebook is answered in **pure SQL** (DuckDB over the cleaned
Parquet from Phase 1). Goal: understand *where* churn concentrates and *how much
revenue* is at stake before any modeling.

**Data:** `data/interim/churn_clean.parquet` — 10,000 customers, 31 columns, cleaned
and schema-validated in Phase 1.

In [1]:
from pathlib import Path

import duckdb

ROOT = Path.cwd()
if not (ROOT / "config").exists():
    ROOT = ROOT.parent
DATA = ROOT / "data" / "interim" / "churn_clean.parquet"

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE VIEW c AS SELECT * FROM read_parquet('{DATA.as_posix()}')")


def q(sql):
    """Run a SQL query and return the result as a DataFrame."""
    return con.sql(sql).df()


q("SELECT COUNT(*) AS rows, COUNT(*) FILTER (churn = 1) AS churned FROM c")

,rows,churned
0,10000,1021


## Q1 — What is the overall churn rate?

In [2]:
q("""
SELECT
    COUNT(*)                        AS customers,
    SUM(churn)                      AS churned,
    ROUND(AVG(churn) * 100, 2)      AS churn_rate_pct
FROM c
""")

,customers,churned,churn_rate_pct
0,10000,1021.0,10.21


**Finding:** 1,021 of 10,000 customers churned — **10.21%**. This is the baseline
every segment below is compared against. It also means a ~9:1 class imbalance for
modeling later.

## Q2 — How much monthly revenue is at risk?

In [3]:
q("""
SELECT
    SUM(monthly_fee) FILTER (churn = 1)                              AS mrr_at_risk,
    SUM(monthly_fee)                                                 AS total_mrr,
    ROUND(100.0 * SUM(monthly_fee) FILTER (churn = 1)
              / SUM(monthly_fee), 2)                                 AS pct_of_mrr
FROM c
""")

,mrr_at_risk,total_mrr,pct_of_mrr
0,35300.0,349300.0,10.11


In [4]:
q("""
SELECT
    customer_segment,
    SUM(monthly_fee) FILTER (churn = 1) AS mrr_at_risk,
    ROUND(AVG(churn) * 100, 2)          AS churn_rate_pct
FROM c
GROUP BY customer_segment
ORDER BY mrr_at_risk DESC
""")

,customer_segment,mrr_at_risk,churn_rate_pct
0,Individual,20810.0,9.98
1,SME,11400.0,10.93
2,Enterprise,3090.0,9.42


**Finding:** **$35,300/month of MRR is walking out the door** (10.1% of $349,300
total). Individuals account for the largest absolute loss ($20,810) simply because they
are the largest segment — churn *rates* are nearly identical across segments (Q3).

## Q3 — Does churn differ by segment, contract, or payment method?

In [5]:
for col in ["customer_segment", "contract_type", "payment_method"]:
    display(q(f"""
        SELECT {col}, COUNT(*) AS n, ROUND(AVG(churn) * 100, 2) AS churn_rate_pct
        FROM c GROUP BY {col} ORDER BY churn_rate_pct DESC
    """))

,customer_segment,n,churn_rate_pct
0,SME,3029,10.93
1,Individual,5984,9.98
2,Enterprise,987,9.42


,contract_type,n,churn_rate_pct
0,Yearly,1983,10.34
1,Monthly,4967,10.33
2,Quarterly,3050,9.93


,payment_method,n,churn_rate_pct
0,Card,5955,10.39
1,PayPal,2557,9.97
2,Bank Transfer,1488,9.88


**Finding:** essentially **no signal** — all groups sit within ~9.4–10.9%, close to
the 10.2% baseline. Contract type surprisingly doesn't matter here (in real telecom data,
monthly contracts usually churn far more). Knowing what *doesn't* drive churn is as
important as knowing what does.

## Q4 — Does the customer lifecycle matter? (tenure cohorts)

In [6]:
q("""
SELECT
    CASE WHEN tenure_months <=  6 THEN '01-06 mo'
         WHEN tenure_months <= 12 THEN '07-12 mo'
         WHEN tenure_months <= 24 THEN '13-24 mo'
         WHEN tenure_months <= 36 THEN '25-36 mo'
         ELSE '37+ mo' END              AS tenure_cohort,
    COUNT(*)                            AS n,
    ROUND(AVG(churn) * 100, 2)          AS churn_rate_pct
FROM c
GROUP BY tenure_cohort
ORDER BY tenure_cohort
""")

,tenure_cohort,n,churn_rate_pct
0,01-06 mo,1032,28.10
1,07-12 mo,1018,8.84
2,13-24 mo,1987,8.76
3,25-36 mo,2049,7.86
4,37+ mo,3914,7.82


**Finding: the single strongest pattern in the data.** Customers in their **first 6
months churn at 28.1%** — nearly 3× baseline — then the rate collapses to ~8–9% and keeps
declining with tenure. This is the classic *onboarding cliff*: survive the first half-year
and you're probably staying. Retention lever → invest in onboarding, not win-back.

## Q5 — Do payment failures push customers out?

In [7]:
q("""
SELECT
    payment_failures,
    COUNT(*)                   AS n,
    ROUND(AVG(churn) * 100, 2) AS churn_rate_pct
FROM c
GROUP BY payment_failures
ORDER BY payment_failures
""")

,payment_failures,n,churn_rate_pct
0,0,6084,8.76
1,1,2989,8.67
2,2,780,25.26
3,3,130,21.54
4,4,14,21.43
5,5,3,33.33


**Finding: a sharp threshold effect.** 0–1 failures ≈ 8.7% churn; **2+ failures jumps
to 21–33%**. One failed payment is noise; two is a red flag. Retention lever → dunning
flows and payment-method recovery outreach triggered at the second failure.

## Q6 — Does recent activity predict staying? (login recency)

In [8]:
q("""
SELECT
    CASE WHEN last_login_days_ago <=  7 THEN '0-7 days'
         WHEN last_login_days_ago <= 14 THEN '8-14 days'
         WHEN last_login_days_ago <= 30 THEN '15-30 days'
         WHEN last_login_days_ago <= 60 THEN '31-60 days'
         ELSE '61+ days' END             AS recency,
    COUNT(*)                             AS n,
    ROUND(AVG(churn) * 100, 2)           AS churn_rate_pct
FROM c
GROUP BY recency
ORDER BY recency
""")

,recency,n,churn_rate_pct
0,0-7 days,5480,9.95
1,15-30 days,1766,9.29
2,31-60 days,410,19.51
3,61+ days,16,18.75
4,8-14 days,2328,9.84


**Finding:** churn is flat (~9–10%) for anyone active in the last 30 days, then
**doubles to ~19.5% past 30 days of silence**. The 30-day mark is a natural trigger for a
re-engagement campaign.

## Q7 — Support: does ticket volume or satisfaction drive churn?

In [9]:
display(q("""
SELECT
    CASE WHEN support_tickets = 0 THEN '0'
         WHEN support_tickets <= 2 THEN '1-2'
         ELSE '3+' END              AS tickets,
    COUNT(*)                        AS n,
    ROUND(AVG(churn) * 100, 2)      AS churn_rate_pct
FROM c GROUP BY tickets ORDER BY tickets
"""))
display(q("""
SELECT csat_score, COUNT(*) AS n, ROUND(AVG(churn) * 100, 2) AS churn_rate_pct
FROM c GROUP BY csat_score ORDER BY csat_score
"""))
display(q("""
SELECT complaint_type, COUNT(*) AS n, ROUND(AVG(churn) * 100, 2) AS churn_rate_pct
FROM c GROUP BY complaint_type ORDER BY churn_rate_pct DESC
"""))

,tickets,n,churn_rate_pct
0,0,3012,10.46
1,1-2,5733,9.91
2,3+,1255,11.00


,csat_score,n,churn_rate_pct
0,1.0,221,23.53
1,2.0,1322,26.02
2,3.0,3380,7.93
3,4.0,3523,6.73
4,5.0,1554,7.72


,complaint_type,n,churn_rate_pct
0,Technical,3498,10.32
1,None,2045,10.32
2,Billing,2427,10.18
3,Service,2030,9.95


**Finding — the most interesting split in the dataset:** ticket *volume* and complaint
*type* are flat (~10%), but **CSAT ≤ 2 churns at 24–26% vs ~7–8% for CSAT ≥ 3**. Customers
don't leave because they contacted support — they leave because support *didn't resolve
their problem well*. Retention lever → CSAT-triggered recovery, not ticket-count alerts.

## Q8 — Do marketing/satisfaction surveys add signal? (NPS, survey response)

In [10]:
display(q("""
SELECT
    CASE WHEN nps_score < 0 THEN 'negative'
         WHEN nps_score <= 50 THEN '0-50'
         ELSE '51-100' END          AS nps_band,
    COUNT(*)                        AS n,
    ROUND(AVG(churn) * 100, 2)      AS churn_rate_pct
FROM c GROUP BY nps_band ORDER BY nps_band
"""))
display(q("""
SELECT survey_response, COUNT(*) AS n, ROUND(AVG(churn) * 100, 2) AS churn_rate_pct
FROM c GROUP BY survey_response ORDER BY churn_rate_pct DESC
"""))

,nps_band,n,churn_rate_pct
0,0-50,4827,10.69
1,51-100,2164,9.84
2,negative,3009,9.70


,survey_response,n,churn_rate_pct
0,Unsatisfied,2047,11.14
1,Satisfied,4975,10.13
2,Neutral,2978,9.70


**Finding:** both nearly flat (9.7–11.1%). In this dataset **NPS does not predict
churn while CSAT strongly does** — a good reminder that "satisfaction" metrics are not
interchangeable, and stated sentiment (surveys) often lags behaviour.

## Q9 — Do discounts or recent price increases matter?

In [11]:
q("""
SELECT
    discount_applied,
    price_increase_last_3m,
    COUNT(*)                    AS n,
    ROUND(AVG(churn) * 100, 2)  AS churn_rate_pct
FROM c
GROUP BY discount_applied, price_increase_last_3m
ORDER BY discount_applied, price_increase_last_3m
""")

,discount_applied,price_increase_last_3m,n,churn_rate_pct
0,0,0,5610,10.48
1,0,1,1340,10.45
2,1,0,2445,9.45
3,1,1,605,10.25


**Finding:** no meaningful effect (9.5–10.5% across all four combinations). Pricing
levers appear churn-neutral in this data.

## Q10 — Sanity check: is geography real signal or synthetic noise?

In [12]:
display(q("""
SELECT
    COUNT(*)         AS country_city_pairs,
    MIN(n)           AS min_pair_count,
    MAX(n)           AS max_pair_count
FROM (SELECT country, city, COUNT(*) AS n FROM c GROUP BY country, city)
"""))
display(q("""
SELECT country, COUNT(*) AS n, ROUND(AVG(churn) * 100, 2) AS churn_rate_pct
FROM c GROUP BY country ORDER BY churn_rate_pct DESC
"""))

,country_city_pairs,min_pair_count,max_pair_count
0,49,180,236


,country,n,churn_rate_pct
0,Germany,1367,11.49
1,Australia,1400,10.86
2,UK,1382,10.35
3,Canada,1488,10.22
4,USA,1442,10.12
5,Bangladesh,1494,9.37
6,India,1427,9.18


**Finding:** all **49 of 49 possible country×city combinations exist** with
near-uniform counts (180–236) — e.g. "Bangladesh / London". The two columns were generated
independently, so geography is **noise, not signal** (country churn rates confirm: all
9.2–11.5%). We will not engineer geographic features. Catching this prevents a model from
memorising fake geography.

---
## Key takeaways

| Driver | Effect | Retention lever |
|---|---|---|
| Tenure ≤ 6 months | **28.1%** churn (3× baseline) | Onboarding program |
| Payment failures ≥ 2 | **21–33%** churn | Dunning / payment recovery at 2nd failure |
| CSAT ≤ 2 | **24–26%** churn | CSAT-triggered save outreach |
| Inactive > 30 days | **~19.5%** churn | 30-day re-engagement campaign |
| Segment, contract, NPS, tickets, pricing, geography | flat ≈ 10% | — (no action) |

**$35.3k/month MRR at risk.** The churn story is behavioural (lifecycle, billing friction,
bad support outcomes, disengagement) — not demographic.